# 2. Private Access — Service Endpoints, Private Endpoints, Private Link

This is one of the most confusing areas on AZ-500. There are three ways to access Azure PaaS services privately, and the exam loves asking which one to use.

## The three options compared

| | Service Endpoint | Private Endpoint | Private Link Service |
|-|-----------------|-----------------|---------------------|
| **What** | Optimized route from VNet to PaaS | Private IP in your VNet for PaaS | Expose YOUR service behind a load balancer |
| **Traffic path** | Still goes to public IP (via MS backbone) | Goes to private IP in your subnet | Goes to private IP via Private Endpoint |
| **DNS** | Uses public FQDN | Custom DNS (privatelink zone) | Custom DNS |
| **Cross-VNet** | No (VNet-local only) | Yes (via VNet peering) | Yes (even cross-tenant) |
| **On-prem access** | No | Yes (via VPN/ExpressRoute) | Yes |
| **NSG support** | No (service endpoints bypass NSGs) | Yes | Yes |
| **Cost** | Free | ~$7.30/month + data | ~$7.30/month + data |
| **Use when** | Quick win, single-VNet | Zero-trust, multi-VNet, on-prem | Exposing your own service to consumers |

## Before you run this notebook

1. Run `uv sync` from the lab folder (`security-certs/az-500/02-networking`).
2. In VS Code, click the **kernel picker** at the top-right of this notebook and choose the interpreter from `.venv` (the one created by `uv`).
3. If the kernel isn't listed, reload the window (`Cmd+Shift+P` then *Reload Window*).

No Docker or Azure subscription is needed — everything is simulated in plain Python so you can learn the concepts safely.

In [ ]:
import json

# Decision tree: which private access method to use?
def recommend_private_access(scenario: dict) -> dict:
    needs_onprem = scenario.get('on_prem_access', False)
    needs_cross_vnet = scenario.get('cross_vnet', False)
    needs_nsg = scenario.get('nsg_enforcement', False)
    is_own_service = scenario.get('own_service', False)
    budget_sensitive = scenario.get('budget_sensitive', False)
    
    if is_own_service:
        return {'recommendation': 'Private Link Service', 'reason': 'Exposing your own service to consumers via Private Endpoint'}
    if needs_onprem or needs_cross_vnet or needs_nsg:
        return {'recommendation': 'Private Endpoint', 'reason': f'Required for: {"on-prem" if needs_onprem else ""} {"cross-VNet" if needs_cross_vnet else ""} {"NSG enforcement" if needs_nsg else ""}'.strip()}
    if budget_sensitive:
        return {'recommendation': 'Service Endpoint', 'reason': 'Free, single-VNet, quick to set up'}
    return {'recommendation': 'Private Endpoint', 'reason': 'Best practice for zero-trust (default recommendation)'}

scenarios = [
    {'name': 'Storage account accessed from one VNet only', 'budget_sensitive': True},
    {'name': 'SQL Database accessed from VNet + on-prem', 'on_prem_access': True},
    {'name': 'Key Vault with NSG restrictions', 'nsg_enforcement': True},
    {'name': 'Multi-VNet hub-spoke accessing Storage', 'cross_vnet': True},
    {'name': 'SaaS vendor exposing API to customers', 'own_service': True},
    {'name': 'Zero-trust architecture for all PaaS', },
]

print('=== Private Access Decision Guide ===\n')
for s in scenarios:
    result = recommend_private_access(s)
    print(f'📋 {s["name"]}')
    print(f'   → {result["recommendation"]} — {result["reason"]}\n')

## 🚫 Bad → ✅ Best: Accessing a Storage Account from a VNet

| Stage | Configuration | What can attack it |
|-------|---------------|--------------------|
| 🚫 **Bad** | Storage account default — public endpoint open | Whole internet (if key leaks, game over) |
| ⚠️ **Okay** | Firewall rules: allow office IP range | Still public FQDN, SAS exfil possible |
| 👍 **Better** | Service Endpoint from `sn-app` subnet + `--default-action Deny` | VNet-local only, traffic still traverses public IP via MS backbone |
| ✅ **Best** | Private Endpoint in `sn-private-endpoints` + privatelink DNS zone | Private IP only; works from peered VNets & on-prem; NSG-enforceable |

The next cell models each stage and shows which callers succeed.

In [ ]:
# Simulate four stages of locking down a Storage Account.
CALLERS = [
    ("Attacker on the internet",         "198.51.100.9",  "public"),
    ("Office laptop",                    "192.0.2.44",    "public"),
    ("VM in sn-app (same VNet)",         "10.0.2.10",     "vnet:sn-app"),
    ("VM in peered spoke VNet",          "10.1.1.10",     "vnet:spoke2"),
    ("On-prem server via VPN",           "172.16.5.20",   "onprem"),
]

def bad(caller):          # Public endpoint, no rules
    return True
def okay(caller):         # Firewall allows 192.0.2.0/24
    return caller[1].startswith("192.0.2.")
def better(caller):       # Service Endpoint from sn-app only
    return caller[2] == "vnet:sn-app"
def best(caller):         # Private Endpoint reachable from any routed network
    return caller[2] in {"vnet:sn-app", "vnet:spoke2", "onprem"}

stages = [("🚫 Bad (public)", bad), ("⚠️  Okay (IP FW)", okay),
          ("👍 Better (SvcEP)", better), ("✅ Best (PrivEP)", best)]

print(f'{"Caller":<30}' + "".join(f"{label:<20}" for label, _ in stages))
print("-" * 110)
for c in CALLERS:
    row = f"{c[0]:<30}"
    for _, fn in stages:
        row += ("✅ allow           " if fn(c) else "🚫 deny            ")
    print(row)

print("\nThe attacker is blocked from stage 'Okay' onwards; only 'Best' also works from peered VNets and on-prem.")

## Why DNS is the hard part of Private Endpoints

Your app connects using the public FQDN (e.g., `mysa.blob.core.windows.net`). A Private Endpoint only helps if that name resolves to the **private IP** in your VNet. If DNS isn't wired up, traffic silently falls back to the public internet.

The cell below simulates the DNS resolution chain so you can see where it breaks.

In [ ]:
# Simulate privatelink DNS resolution.
PUBLIC_DNS = {
    "mysa.blob.core.windows.net": "52.239.130.5",          # Azure public IP
}
# A CNAME always exists: <name>.blob.core.windows.net -> <name>.privatelink.blob.core.windows.net
PUBLIC_CNAMES = {
    "mysa.blob.core.windows.net": "mysa.privatelink.blob.core.windows.net",
}
# Private DNS zone, linked to VNet, overrides the privatelink.* name.
PRIVATE_DNS_ZONE = {
    "mysa.privatelink.blob.core.windows.net": "10.0.4.5",  # Private Endpoint NIC IP
}

def resolve(name, *, private_zone_linked):
    trace = [f"query: {name}"]
    # Follow CNAME if present
    if name in PUBLIC_CNAMES:
        target = PUBLIC_CNAMES[name]
        trace.append(f"  CNAME -> {target}")
        name = target
    if private_zone_linked and name in PRIVATE_DNS_ZONE:
        trace.append(f"  Private DNS zone -> {PRIVATE_DNS_ZONE[name]}  (PRIVATE IP ✅)")
        return PRIVATE_DNS_ZONE[name], trace
    if name in PUBLIC_DNS:
        trace.append(f"  Public DNS -> {PUBLIC_DNS[name]}  (PUBLIC IP 🚫)")
        return PUBLIC_DNS[name], trace
    # Fall back to the original public record if CNAME has no private override
    for k, v in PUBLIC_DNS.items():
        if PUBLIC_CNAMES.get(k) == name:
            trace.append(f"  Public DNS fallback -> {v}  (PUBLIC IP 🚫)")
            return v, trace
    return None, trace + ["  NXDOMAIN"]

for label, linked in [("❌ Private DNS zone NOT linked to VNet", False),
                      ("✅ Private DNS zone linked to VNet",     True)]:
    print(f"--- {label} ---")
    ip, trace = resolve("mysa.blob.core.windows.net", private_zone_linked=linked)
    for line in trace:
        print(line)
    print(f"Result: {ip}\n")

## VNet peering and VPN gateways

Private Endpoints are useful only if your network can **reach** the VNet that hosts them. The two primary connectivity options on AZ-500 are:

### VNet peering

- Low-latency, private connection between two VNets over the Microsoft backbone.
- **Not transitive by default**: if A ↔ B and B ↔ C are peered, A still cannot reach C unless you add an NVA/route or use *gateway transit*.
- Use for **hub-and-spoke**: all spokes peer with the hub; the hub runs a firewall/NVA; UDRs send spoke traffic through the hub.

```bash
# Peer spoke -> hub, allowing spoke to use hub's gateway
az network vnet peering create -g rg-prod -n spoke1-to-hub \
  --vnet-name vnet-spoke1 --remote-vnet vnet-hub \
  --allow-vnet-access --allow-forwarded-traffic --use-remote-gateways

# Peer hub -> spoke, sharing the hub's gateway
az network vnet peering create -g rg-prod -n hub-to-spoke1 \
  --vnet-name vnet-hub --remote-vnet vnet-spoke1 \
  --allow-vnet-access --allow-forwarded-traffic --allow-gateway-transit
```

### VPN gateway (Site-to-Site)

Connects your on-prem network to Azure over an encrypted IPsec tunnel.

| SKU | Aggregate throughput | Use for |
|-----|---------------------|---------|
| `Basic` | 100 Mbps | Dev/test only, no SLA |
| `VpnGw1`–`VpnGw5` | 650 Mbps – 10 Gbps | Production S2S |
| `VpnGw*AZ` | Same + zone-redundant | Production requiring HA |

For faster / private connections, use **ExpressRoute** (private circuit from a carrier, up to 100 Gbps).

The cell below simulates transitivity so you can see why spoke-to-spoke traffic needs the hub.

In [ ]:
# Simulate VNet peering reachability, including the "not transitive" rule.
PEERINGS = {
    ("vnet-hub", "vnet-spoke1"),
    ("vnet-hub", "vnet-spoke2"),
    # NOTE: no direct peering between spoke1 and spoke2.
}
# Normalize to a set of unordered pairs for lookup
PAIRS = {frozenset(p) for p in PEERINGS}

def directly_peered(a, b):
    return frozenset({a, b}) in PAIRS

def reachable(src, dst, *, hub=None):
    if src == dst:
        return True, "same VNet"
    if directly_peered(src, dst):
        return True, "direct peering"
    if hub and directly_peered(src, hub) and directly_peered(hub, dst):
        return True, f"via {hub} (NVA/firewall + UDR required — peering alone is NOT transitive)"
    return False, "no route"

tests = [
    ("vnet-spoke1", "vnet-hub"),
    ("vnet-spoke1", "vnet-spoke2"),   # Needs hub transit
    ("vnet-spoke2", "vnet-spoke1"),
]
for a, b in tests:
    ok, why = reachable(a, b, hub="vnet-hub")
    icon = "✅" if ok else "🚫"
    print(f"{icon} {a} -> {b}: {why}")

## Service Endpoints — implementation

```bash
# Enable service endpoint for Storage on a subnet
az network vnet subnet update -g rg-prod --vnet-name vnet-prod \
  -n sn-app --service-endpoints Microsoft.Storage

# Lock down storage account to only accept traffic from that subnet
az storage account network-rule add -g rg-prod -n mystorageaccount \
  --vnet-name vnet-prod --subnet sn-app

# Set default action to deny (block all except VNet rules)
az storage account update -g rg-prod -n mystorageaccount \
  --default-action Deny
```

## Private Endpoints — implementation

```bash
# Create Private Endpoint for a Storage Account
az network private-endpoint create -g rg-prod -n pe-storage \
  --vnet-name vnet-prod --subnet sn-private-endpoints \
  --private-connection-resource-id /subscriptions/.../storageAccounts/mysa \
  --group-id blob --connection-name mysa-blob-conn

# Create Private DNS Zone (so mysa.blob.core.windows.net resolves to private IP)
az network private-dns zone create -g rg-prod -n privatelink.blob.core.windows.net

# Link DNS zone to VNet
az network private-dns link vnet create -g rg-prod \
  --zone-name privatelink.blob.core.windows.net \
  --name link-vnet-prod --virtual-network vnet-prod \
  --registration-enabled false

# Create DNS record group
az network private-endpoint dns-zone-group create -g rg-prod \
  --endpoint-name pe-storage --name default \
  --private-dns-zone privatelink.blob.core.windows.net \
  --zone-name blob
```

### DNS is the hard part

After creating a Private Endpoint, `mysa.blob.core.windows.net` must resolve to the **private IP** (e.g., 10.0.4.5), not the public IP. This requires:
1. A Private DNS zone (`privatelink.blob.core.windows.net`)
2. Linked to your VNet
3. An A record pointing to the PE's private IP

If DNS isn't configured, your app still resolves the public IP and the traffic goes through the internet.

## App Service VNet Integration

App Service (and Functions) can be integrated with a VNet for **outbound** traffic:

```bash
# Integrate App Service with a VNet (outbound traffic goes through VNet)
az webapp vnet-integration add -g rg-prod -n my-webapp \
  --vnet vnet-prod --subnet sn-app-integration
```

For **inbound** private access to App Service, use a Private Endpoint:

```bash
az network private-endpoint create -g rg-prod -n pe-webapp \
  --vnet-name vnet-prod --subnet sn-private-endpoints \
  --private-connection-resource-id /subscriptions/.../sites/my-webapp \
  --group-id sites --connection-name webapp-conn
```

---
## Summary

| Implementation | When to use |
|---------------|-------------|
| **Service Endpoint** | Simple, single-VNet, free. No on-prem access. |
| **Private Endpoint** | Zero-trust default. Private IP in your VNet. Needs DNS config. |
| **Private Link Service** | You're exposing your own service to external consumers. |
| **App Service VNet integration** | Outbound: VNet integration. Inbound: Private Endpoint. |
| **DNS zones** | `privatelink.<service>.core.windows.net` — link to VNet. |

**Next**: [Notebook 3 — Public Access and Firewalls](03_public_access_and_firewalls.ipynb)